PySpark Interview Questions 

Day 1 — Scenario-Based Scenario

Question :
You have an employee attendance DataFrame:

attendance_df

| Column Name | Type |
| ------------- | ---------------------------- |
| `emp_id` | string |
| `login_time` | string (yyyy-MM-dd HH:mm:ss) |
| `logout_time` | string (yyyy-MM-dd HH:mm:ss) |
| `location` | string |

Scenario Requirements:

Your manager wants an Employee Daily Working Hours Report:

1. Convert `login_time` and `logout_time` to proper timestamp.
2. Calculate working_hours = difference between logout and login in hours (decimal format).
3. Extract date from login_time.
4. Generate daily summary:

* total_hours_worked per employee per day
* first_login_time
* last_logout_time
5. Output columns:
`emp_id, date, total_hours_worked, first_login_time, last_logout_time`
6. Sort by `emp_id`, `date`.

Provide a single combined PySpark solution.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

data = [
    ('EMP1001', '2026-05-19 08:45:12', '2026-05-19 17:32:48', 'Hyderabad'),
    ('EMP1002', '2026-05-19 09:10:05', '2026-05-19 18:05:27', 'Bengaluru'),
    ('EMP1003', '2026-05-19 07:58:44', '2026-05-19 16:49:11', 'Chennai'),
    ('EMP1004', '2026-05-19 10:02:19', '2026-05-19 19:15:36', 'Mumbai'),
    ('EMP1005', '2026-05-19 08:20:30', '2026-05-19 17:00:09', 'Pune'),
    ('EMP1006', '2026-05-19 09:35:50', '2026-05-19 18:22:14', 'Visakhapatnam'),
    ('EMP1001', '2026-02-19 08:45:12', '2026-03-19 17:32:48', 'Hyderabad'),
    ('EMP1002', '2026-05-19 09:10:05', '2026-06-19 18:05:27', 'Bengaluru'),
]

schema=StructType([
    StructField('emp_id',StringType(),False ),
    StructField('login_time',StringType(),True),
    StructField('logout_time',StringType(),True),
    StructField('location',StringType(), True)
])

attendance_df=spark.createDataFrame(data,schema)
#PySpark cannot directly subtract string timestamps, so we first convert them into Unix timestamps,
#This converts the datetime string into seconds from Jan 1, 1970 (Epoch time).
#Example:2026-05-19 08:45:12'→ 1779176712, 1 hour = 3600 seconds
attendance_df = attendance_df.withColumn(
    "date",
    to_date("login_time"))
windowSpec=Window.partitionBy('emp_id','date').orderBy('emp_id','date')
attendance_df=attendance_df.withColumn('login_time', to_timestamp(col('login_time')))
attendance_df=attendance_df.withColumn('login_time', to_timestamp(col('login_time'))).withColumn('logout_time', to_timestamp(col('logout_time'))).withColumn('working_hours', (((unix_timestamp(col('logout_time')) - unix_timestamp(col('login_time'))) / 3600)).cast('decimal(10,2)')).withColumn('first_login_time',to_date(min('login_time').over(windowSpec))).withColumn('last_logout_time',to_date(max('logout_time').over(windowSpec)))
attendance_df=attendance_df.withColumn('total_hours_worked',sum(col('working_hours')).over(windowSpec)).drop('login_time','logout_time','working_hours','location').show(truncate=False)
